# SQL Demo: The Northwind Trading Company 🛒

**Learning Objective:** By the end of this notebook, you will be able to use SQL to ask and answer real business questions about customers, products, and orders.

**Economics Concept:** We will explore a company's sales database to understand who buys what, where customers come from, and which products earn the most revenue.

---

## What is the Northwind Database?

Northwind is a fictional trading company that buys food products from suppliers around the world and sells them to customers in many countries.

Think of it like a small international grocery wholesaler.

The database has tables for:
- **Products** — what the company sells
- **Customers** — who buys from them
- **Orders** — each purchase made
- **Employees** — the people who work there
- **Suppliers** — the companies that provide the products
- **Categories** — the types of food (like Beverages, Dairy, Seafood)

This is real-world data! The same kind of database structure is used by companies everywhere.

## Step 1: Set Up Our Tools

Before we can talk to the database, we need to load some Python libraries.

A **library** is a collection of pre-written code that gives us extra powers.

Run the cell below to install the SQL libraries if you haven't already.

In [ ]:
#!pip install sqlalchemy jupysql

Now we import the libraries we need.

Each line below loads one library and has a comment explaining what it does.

In [ ]:
import pandas as pd        # pandas: lets us work with tables of data in Python
import sqlalchemy          # sqlalchemy: lets Python talk to databases
from pathlib import Path   # pathlib: helps us point to files on the computer
%load_ext sql              # this loads the SQL magic commands so we can type SQL directly in cells

## Step 2: Connect to the Database

A **database** is a structured collection of data stored in a file.

We use a **connection** to open the database and start asking it questions.

Think of it like opening a spreadsheet file — you need to open it before you can read it.

In [ ]:
# Create the connection to the northwind database file
northwind_engine = sqlalchemy.create_engine('sqlite:///northwind.db')
northwind_connection = northwind_engine.connect()

Now we tell the SQL magic to use that connection.

After this cell runs, every `%sql` or `%%sql` cell will talk to the Northwind database.

In [ ]:
%sql northwind_engine

## Step 3: Explore the Database

Before we ask questions, let's see what tables are in the database.

A **table** is like one sheet in a spreadsheet. Each table stores information about one thing (products, customers, orders, etc.).

The query below asks SQLite to list all tables. `sqlite_master` is a special hidden table that SQLite uses to keep track of everything in the database.

In [ ]:
%%sql
SELECT name
FROM sqlite_master
WHERE type = 'table';

You can see all the tables in the database. We have tables for Categories, Customers, Employees, Orders, Products, Suppliers, and more!

Now let's look at the **Products** table first. This is the heart of the business — it tells us what the company sells.

In [ ]:
%sql SELECT * FROM Products LIMIT 10;

The `SELECT *` command means "give me all columns." The `LIMIT 10` means "show only the first 10 rows" — useful when a table has thousands of rows.

You can see each product has a name, a price (`UnitPrice`), and a stock level (`UnitsInStock`).

Let's check the column names more carefully using `PRAGMA table_info`. This tells us the exact names and data types of every column.

In [ ]:
%sql PRAGMA table_info(Products);

## Step 4: SELECT — Choose Which Columns You Want

Instead of showing every column, we can ask for just the columns we care about.

This is useful when a table has many columns and you only need a few.

Let's get just the product name and price.

In [ ]:
%sql SELECT ProductName, UnitPrice FROM Products;

77 products! That's a lot of food to keep track of.

## Step 5: WHERE — Filter Rows by a Condition

A **filter** narrows down the results so you only see rows that match a condition.

In economics, we often filter data — for example, "show me only workers who earn above the median wage."

Here, let's find all products that cost more than $50. This tells us which products are in the premium price range.

In [ ]:
%%sql
SELECT ProductName, UnitPrice, UnitsInStock
FROM Products
WHERE UnitPrice > 50;

These are the high-end products. Notice how most are specialty items — things like luxury teas or imported meats.

Now let's find products that are **out of stock**. When `UnitsInStock` equals 0, the shelf is empty.

In business, running out of stock means lost sales — an important signal for managers.

In [ ]:
%%sql
SELECT ProductName, UnitPrice, UnitsInStock
FROM Products
WHERE UnitsInStock = 0;

## Step 6: ORDER BY — Sort the Results

**Sorting** arranges the rows in a specific order, from highest to lowest or A to Z.

Let's find the most expensive products by sorting by price from high to low.

The keyword `DESC` means descending — from the highest value down to the lowest.

In [ ]:
%%sql
SELECT ProductName, UnitPrice
FROM Products
ORDER BY UnitPrice DESC;

The most expensive product is Côte de Blaye at over $260! That's a very fancy wine.

Now let's flip it and find the cheapest products. Use `ASC` for ascending order.

In [ ]:
%%sql
SELECT ProductName, UnitPrice
FROM Products
ORDER BY UnitPrice ASC
LIMIT 10;

## Step 7: Meet the Customers

Now let's explore who buys from Northwind.

The **Customers** table tells us about every company that has placed an order. These customers come from all over the world!

In [ ]:
%sql SELECT CustomerID, CompanyName, City, Country FROM Customers LIMIT 10;

We can see companies from Germany, Mexico, UK, and more.

Let's find all customers from the **USA**. In economics, we might ask: which market segment is domestic versus international?

In [ ]:
%%sql
SELECT CompanyName, City, Country
FROM Customers
WHERE Country = 'USA';

## Step 8: COUNT — How Many Rows Match?

`COUNT(*)` counts how many rows are in the result. It's one of the most useful tools in SQL.

Let's count how many customers are in the entire database.

In [ ]:
%sql SELECT COUNT(*) AS total_customers FROM Customers;

93 customers worldwide! 

The word `AS` in the query creates an **alias** — a friendly label for the result column. Instead of `COUNT(*)`, we called it `total_customers` so it's easier to read.

## Step 9: GROUP BY — Count by Category

`GROUP BY` is one of the most powerful tools in SQL.

It groups rows together by a shared value, then lets you count or sum within each group.

Think of it like asking: "How many customers do we have in each country?"

In economics, GROUP BY is how we compute statistics for different subgroups — for example, unemployment rates by education level.

In [ ]:
%%sql
SELECT Country, COUNT(*) AS number_of_customers
FROM Customers
GROUP BY Country
ORDER BY number_of_customers DESC;

The USA, Germany, and France are the biggest customer markets. This kind of insight helps a company decide where to focus its sales team!

Now let's count how many products are in each food category.

In [ ]:
%%sql
SELECT CategoryID, COUNT(*) AS number_of_products
FROM Products
GROUP BY CategoryID
ORDER BY number_of_products DESC;

We see the category ID numbers but not the names. Let's fix that with a JOIN!

## Step 10: JOIN — Combine Two Tables

A **JOIN** connects two tables by matching rows that share a common value.

Here's the idea:
- The `Products` table has a column called `CategoryID` (a number like 1, 2, 3)
- The `Categories` table also has `CategoryID`, plus the real category names

When we JOIN these tables on `CategoryID`, each product row gets the category name attached to it.

Think of it like looking up a code in a reference book to find the full description.

In [ ]:
%%sql
SELECT Products.ProductName, Categories.CategoryName, Products.UnitPrice
FROM Products
JOIN Categories ON Products.CategoryID = Categories.CategoryID
LIMIT 15;

Now we can see the real category name next to each product. Much easier to read!

Let's now count how many products are in each named category, combining GROUP BY and JOIN together.

In [ ]:
%%sql
SELECT Categories.CategoryName, COUNT(*) AS number_of_products
FROM Products
JOIN Categories ON Products.CategoryID = Categories.CategoryID
GROUP BY Categories.CategoryName
ORDER BY number_of_products DESC;

Confections (sweets and candy) and Beverages have the most product variety. Seafood has the fewest.

This kind of analysis helps a business decide where to expand its product line.

## Step 11: The Employees

Let's meet the people who work at Northwind. The **Employees** table has information about the sales team.

Northwind has a small but global team — based in both the USA and the UK.

In [ ]:
%%sql
SELECT FirstName, LastName, Title, City, Country
FROM Employees;

Only 9 employees, and most are Sales Representatives. Andrew Fuller is the Vice President of Sales — he's the boss!

The `ReportsTo` column in the Employees table tells us who each person reports to. Let's use a JOIN to see the manager's name next to each employee.

This is called a **self-join** — we join the Employees table to itself using two different aliases.

In [ ]:
%%sql
SELECT
    emp.FirstName || ' ' || emp.LastName AS employee_name,
    emp.Title AS employee_title,
    mgr.FirstName || ' ' || mgr.LastName AS reports_to_manager
FROM Employees AS emp
LEFT JOIN Employees AS mgr ON emp.ReportsTo = mgr.EmployeeID;

A `LEFT JOIN` includes all rows from the left table, even if there is no match in the right table. Andrew Fuller has no manager listed — that's because he's at the top!

The `||` operator is how SQLite **concatenates** (joins together) text. We used it to combine `FirstName` and `LastName` into a full name.

## Step 12: Orders — The Business in Action

Now let's look at the **Orders** table. Every time a customer buys something, a new row appears here.

This is where all the business action happens!

In [ ]:
%sql SELECT * FROM Orders LIMIT 10;

Each order has an ID, a customer, an employee who handled it, and dates. The `Freight` column is the shipping cost.

Let's count the total number of orders in the database.

In [ ]:
%sql SELECT COUNT(*) AS total_orders FROM Orders;

Over 16,000 orders! That's a busy company.

Now let's JOIN Orders with Customers to see which companies placed the most orders.

**Hypothesis:** The largest customers probably place the most orders. This is a key idea in business — a small number of customers often accounts for most of the activity.

In [ ]:
%%sql
SELECT Customers.CompanyName, COUNT(Orders.OrderID) AS number_of_orders
FROM Orders
JOIN Customers ON Orders.CustomerID = Customers.CustomerID
GROUP BY Customers.CompanyName
ORDER BY number_of_orders DESC
LIMIT 10;

These are Northwind's top 10 most loyal customers. Save Lot Markets leads the pack!

## Step 13: AVG and SUM — Aggregate Functions

**Aggregate functions** compute a single summary number from many rows.

- `SUM()` adds everything up
- `AVG()` finds the average
- `MAX()` finds the largest value
- `MIN()` finds the smallest value

In economics, we use averages and sums all the time — average wages, total GDP, minimum price, and so on.

Let's find the average price of all products.

In [ ]:
%%sql
SELECT
    AVG(UnitPrice) AS average_product_price,
    MAX(UnitPrice) AS most_expensive_product,
    MIN(UnitPrice) AS cheapest_product
FROM Products;

The average product price is about $29. There's a huge gap between the cheapest ($2.50) and the most expensive ($263.50). That's a sign of high price dispersion.

Now let's see the average price **by category**. We use GROUP BY again to split the average into groups.

In [ ]:
%%sql
SELECT
    Categories.CategoryName,
    ROUND(AVG(Products.UnitPrice), 2) AS average_price_in_category,
    COUNT(Products.ProductID) AS product_count
FROM Products
JOIN Categories ON Products.CategoryID = Categories.CategoryID
GROUP BY Categories.CategoryName
ORDER BY average_price_in_category DESC;

Meat and Poultry, and Produce have the highest average prices. Grains and Condiments are cheapest on average.

The `ROUND()` function rounds the number to 2 decimal places, which is the standard for money.

## Step 14: Save Results to a Pandas DataFrame

Sometimes you want to take SQL results and do more analysis in Python — like making a chart.

We can save a SQL query result into a variable using the `<<` syntax, then convert it to a pandas **DataFrame**.

A **DataFrame** is a table-like structure in Python that lets you do further analysis and create plots.

In [ ]:
%%sql customers_by_country_result <<
SELECT Country, COUNT(*) AS number_of_customers
FROM Customers
GROUP BY Country
ORDER BY number_of_customers DESC;

The `<<` saved the result into a variable called `customers_by_country_result`. Now we convert it to a DataFrame so we can work with it in pandas.

In [ ]:
customers_by_country_data = customers_by_country_result.DataFrame()
customers_by_country_data.head(10)

Now we have a real pandas DataFrame! We can make a quick bar chart to visualize the top 10 customer countries.

In [ ]:
top_10_customer_countries = customers_by_country_data.head(10)
top_10_customer_countries.plot(
    kind='bar',
    x='Country',
    y='number_of_customers',
    title='Number of Northwind Customers by Country (Top 10)',
    legend=False,
    color='steelblue'
)

The chart clearly shows that the USA, Germany, and France are Northwind's biggest markets by number of customers. 

This is a useful insight — if Northwind wants to grow, it should either expand more in the USA (where it's already strong) or find new customers in countries where it has few.

## Step 15: Bonus — Who Are the Top Suppliers?

Northwind gets its products from **Suppliers** around the world.

Let's find which countries supply the most products. This tells us where Northwind's supply chain comes from.

In [ ]:
%%sql
SELECT Suppliers.Country, COUNT(Products.ProductID) AS products_supplied
FROM Products
JOIN Suppliers ON Products.SupplierID = Suppliers.SupplierID
GROUP BY Suppliers.Country
ORDER BY products_supplied DESC;

The USA is both the biggest customer country and the biggest supplier country!

That's an interesting pattern. It suggests Northwind has deep ties to the US market on both sides of the trade — buying and selling.

## Step 16: Using LIKE — Pattern Matching

Sometimes you want to search for rows where a text column matches a pattern, not an exact value.

The `LIKE` keyword lets you do this. The `%` symbol is a **wildcard** — it means "any characters here."

For example, `LIKE 'Ch%'` finds anything that starts with `Ch`.

In [ ]:
%%sql
SELECT ProductName, UnitPrice
FROM Products
WHERE ProductName LIKE 'Ch%';

Everything that starts with "Ch" — including "Chai", "Chang", and various "Chef Anton" products.

You can also search for a pattern anywhere in the name using `%` on both sides.

In [ ]:
%%sql
SELECT ProductName, UnitPrice
FROM Products
WHERE ProductName LIKE '%sauce%';

All products with the word "sauce" anywhere in the name.

## Step 17: Challenge Query — Which Employee Handled the Most Orders?

Let's put it all together: a query that joins three ideas — Orders, Employees, and GROUP BY — to find which employee handled the most business.

This is the kind of question a manager might ask when evaluating staff performance.

In [ ]:
%%sql
SELECT
    Employees.FirstName || ' ' || Employees.LastName AS employee_full_name,
    Employees.Title AS job_title,
    COUNT(Orders.OrderID) AS orders_handled
FROM Orders
JOIN Employees ON Orders.EmployeeID = Employees.EmployeeID
GROUP BY Employees.EmployeeID
ORDER BY orders_handled DESC;

Margaret Peacock handled the most orders by far!

Notice that the number of orders per employee varies a lot. This kind of variation — where some workers produce much more than others — is a common finding in labor economics.

---

## 🎉 Summary: What You Learned

Congratulations! You just ran real SQL queries on a real business database.

Here is what each SQL command does:

| Command | What it does |
|---|---|
| `SELECT` | Chooses which columns to show |
| `FROM` | Names the table to look in |
| `WHERE` | Filters rows by a condition |
| `ORDER BY` | Sorts the results |
| `LIMIT` | Shows only the first N rows |
| `COUNT()` | Counts how many rows |
| `AVG()` | Finds the average |
| `SUM()` | Adds all values together |
| `GROUP BY` | Groups rows and computes summaries per group |
| `JOIN` | Combines two tables by matching a shared column |
| `LIKE` | Finds rows where text matches a pattern |

**What we discovered about Northwind:**
- The USA, Germany, and France are the biggest customer markets
- The most expensive product is Côte de Blaye wine at $263.50
- Confections and Beverages have the most product variety
- Margaret Peacock handled the most customer orders
- The USA is both Northwind's largest customer and supplier country

SQL is the language of data. Almost every company with a database uses it every day. Now you know the basics!